# 🏠 M4 Brief 1 – Benchmark de Modèles de Régression
## Dataset Boston Housing | FastIA 2025

---

**Objectif :** Prédire le prix médian des logements (`medv`) à partir de variables socio-économiques et urbaines.

**Pipeline :**
1. Analyse exploratoire (EDA)
2. Analyse éthique
3. Nettoyage et préprocessing
4. Benchmark : Régression Linéaire, Random Forest, LightGBM
5. Comparaison et interprétation des résultats

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import lightgbm as lgb

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print('✔  Imports OK')

## 1. Chargement des données

In [ ]:
df = pd.read_csv('bostonhousing-693729dedb019653836667.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
# Description des variables
variables = {
    'crim':    'Taux de criminalité par habitant',
    'zn':      'Proportion de terrains résidentiels > 25 000 m²',
    'indus':   'Proportion de zones industrielles non commerciales',
    'chas':    'Variable indicatrice Charles River (1=bordure, 0=non)',
    'nox':     'Concentration en oxydes nitriques (polluant)',
    'rm':      'Nombre moyen de pièces par logement',
    'age':     'Proportion de logements construits avant 1940',
    'dis':     'Distance pondérée aux centres d\'emploi de Boston',
    'rad':     'Indice d\'accessibilité aux autoroutes radiales',
    'tax':     'Taux d\'imposition foncière (par 10 000$)',
    'ptratio': 'Ratio élèves/enseignant par ville',
    'b':       '⚠️ SENSIBLE : proportion résidents noirs (1000(Bk-0.63)²)',
    'lstat':   '% de population à faibles revenus',
    'medv':    '🎯 TARGET : valeur médiane logements (milliers $)'
}
pd.DataFrame.from_dict(variables, orient='index', columns=['Description'])

## 2. ⚖️ Analyse Éthique

### ⚠️ Controverse du Dataset Boston Housing

Ce dataset est **controversé** et a été **retiré de scikit-learn en 2023**. La raison principale :

**La colonne `b`** encode la proportion de résidents noirs avec la formule `1000(Bk - 0.63)²`. Cette variable :
- Introduit une **discrimination raciale directe** dans le modèle
- Peut conduire à des décisions discriminatoires dans l'estimation immobilière
- Viole les principes d'équité en IA et le RGPD (données sensibles)

**Source :** Carlini et al. (2021) – "Racist Data Destruction?"

### Décision éthique
La colonne `b` sera **supprimée** avant tout entraînement de modèle.

In [ ]:
# Analyse de la corrélation de 'b' avec la target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Analyse éthique – Variable b (sensible)', fontsize=13, fontweight='bold')

axes[0].scatter(df['b'], df['medv'], alpha=0.4, color='#E07B54', s=15)
axes[0].set_xlabel('b (proportion résidents noirs)')
axes[0].set_ylabel('medv (prix médian)')
axes[0].set_title(f'Corrélation b/medv : {df["b"].corr(df["medv"]):.3f}')

axes[1].hist(df['b'], bins=30, color='#E07B54', edgecolor='white')
axes[1].set_xlabel('b')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution de b – très asymétrique')

plt.tight_layout()
plt.savefig('graphiques/ethique_variable_b.png', dpi=150)
plt.show()
print('\n❌ Décision : suppression de la colonne b (discrimination raciale)')

## 3. Analyse Exploratoire (EDA)

In [ ]:
# Statistiques descriptives
print('Valeurs manquantes :', df.isnull().sum().sum())
print('Doublons :', df.duplicated().sum())
df.describe().round(2)

In [ ]:
# Distribution de la target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribution de la target (medv)', fontsize=13, fontweight='bold')

sns.histplot(df['medv'], kde=True, ax=axes[0], color='#5B8DB8', edgecolor='white')
axes[0].set_title('Distribution medv')
axes[0].axvline(df['medv'].mean(), color='red', linestyle='--', label=f'Moy. {df["medv"].mean():.1f}')
axes[0].legend()

sns.boxplot(y=df['medv'], ax=axes[1], color='#5B8DB8')
axes[1].set_title('Boxplot medv – valeurs à 50k$ (plafond)')

plt.tight_layout()
plt.savefig('graphiques/distribution_target.png', dpi=150)
plt.show()
print(f'Valeurs à 50k$ (plafond) : {(df["medv"] == 50).sum()} lignes → valeurs censurées')

In [ ]:
# Distributions de toutes les features
fig, axes = plt.subplots(3, 5, figsize=(18, 12))
axes = axes.flatten()
fig.suptitle('Distributions des variables', fontsize=13, fontweight='bold')

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='#5B8DB8', edgecolor='white')
    axes[i].set_title(col, fontweight='bold')

for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('graphiques/distributions.png', dpi=150)
plt.show()

In [ ]:
# Matrice de corrélation
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title('Matrice de corrélation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graphiques/correlation.png', dpi=150)
plt.show()

# Top corrélations avec medv
print('\nTop corrélations avec medv :')
print(corr['medv'].sort_values().to_string())

In [ ]:
# Détection outliers (IQR)
colonnes_num = df.select_dtypes(include=np.number).columns.tolist()
outliers_rapport = {}
for col in colonnes_num:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    nb = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outliers_rapport[col] = {'nb': nb, 'pct': round(nb/len(df)*100, 2)}

pd.DataFrame(outliers_rapport).T.sort_values('pct', ascending=False)

## 4. Nettoyage et Préprocessing

In [ ]:
# Copie de travail
df_clean = df.copy()

# 1. Suppression colonne sensible
df_clean = df_clean.drop(columns=['b'])
print('❌ Colonne b supprimée (discrimination raciale – Art. 9 RGPD)')

# 2. Suppression lignes avec medv == 50 (valeurs censurées/plafonnées)
nb_avant = len(df_clean)
df_clean = df_clean[df_clean['medv'] < 50]
print(f'❌ {nb_avant - len(df_clean)} lignes supprimées (medv=50, valeurs censurées)')

# 3. Winsorisation outliers sur crim (très asymétrique)
Q1, Q3 = df_clean['crim'].quantile(0.25), df_clean['crim'].quantile(0.75)
IQR = Q3 - Q1
df_clean['crim'] = df_clean['crim'].clip(upper=Q3 + 3*IQR)
print('✅ crim : winsorisation IQR × 3 (distribution très asymétrique)')

print(f'\nDataset final : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes')
print(f'NaN restants : {df_clean.isnull().sum().sum()}')

In [ ]:
# Séparation X / y
X = df_clean.drop(columns=['medv'])
y = df_clean['medv']

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} | Test : {X_test.shape[0]}')
print(f'Features : {X.shape[1]}')

## 5. Benchmark des Modèles

In [ ]:
# Configuration cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluer_modele(nom, modele, X_tr, X_te, y_tr, y_te):
    """Entraîne, évalue et retourne les métriques d'un modèle."""
    # Cross-validation
    cv_rmse = np.sqrt(-cross_val_score(
        modele, X_tr, y_tr, cv=kfold,
        scoring='neg_mean_squared_error'
    ))
    cv_r2 = cross_val_score(modele, X_tr, y_tr, cv=kfold, scoring='r2')

    # Entraînement final
    modele.fit(X_tr, y_tr)
    y_pred = modele.predict(X_te)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)

    print(f'\n── {nom} ──')
    print(f'  CV RMSE  : {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}')
    print(f'  CV R²    : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
    print(f'  Test RMSE: {rmse:.3f}')
    print(f'  Test MAE : {mae:.3f}')
    print(f'  Test R²  : {r2:.3f}')

    return {
        'Modèle': nom,
        'CV RMSE (moy)': round(cv_rmse.mean(), 3),
        'CV RMSE (std)': round(cv_rmse.std(), 3),
        'CV R² (moy)':   round(cv_r2.mean(), 3),
        'Test RMSE':     round(rmse, 3),
        'Test MAE':      round(mae, 3),
        'Test R²':       round(r2, 3),
    }, modele, y_pred

print('✔  Fonction d\'évaluation définie')

In [ ]:
# MODÈLE 1 : Régression Linéaire
lr = LinearRegression()
res_lr, model_lr, pred_lr = evaluer_modele(
    'Régression Linéaire', lr, X_train_s, X_test_s, y_train, y_test)

In [ ]:
#MODÈLE 2 : Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
res_rf, model_rf, pred_rf = evaluer_modele(
    'Random Forest', rf, X_train_s, X_test_s, y_train, y_test)

In [ ]:
# MODÈLE 3: LightGBM
lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                          random_state=SEED, verbose=-1)
res_lgbm, model_lgbm, pred_lgbm = evaluer_modele(
    'LightGBM', lgbm, X_train_s, X_test_s, y_train, y_test)

## 6. Comparaison des Résultats

In [ ]:
# Tableau comparatif
resultats = pd.DataFrame([res_lr, res_rf, res_lgbm])
resultats = resultats.set_index('Modèle')
print('\n── Tableau comparatif ──')
resultats

In [ ]:
# Graphiques comparatifs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparaison des modèles', fontsize=13, fontweight='bold')

noms   = ['Régression\nLinéaire', 'Random\nForest', 'LightGBM']
colors = ['#AACCE0', '#5B8DB8', '#028090']

# RMSE
rmses = [res_lr['Test RMSE'], res_rf['Test RMSE'], res_lgbm['Test RMSE']]
b1 = axes[0].bar(noms, rmses, color=colors, edgecolor='white')
axes[0].set_title('Test RMSE (↓ meilleur)', fontweight='bold')
for bar, v in zip(b1, rmses):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}', ha='center', fontsize=11)

# MAE
maes = [res_lr['Test MAE'], res_rf['Test MAE'], res_lgbm['Test MAE']]
b2 = axes[1].bar(noms, maes, color=colors, edgecolor='white')
axes[1].set_title('Test MAE (↓ meilleur)', fontweight='bold')
for bar, v in zip(b2, maes):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}', ha='center', fontsize=11)

# R²
r2s = [res_lr['Test R²'], res_rf['Test R²'], res_lgbm['Test R²']]
b3 = axes[2].bar(noms, r2s, color=colors, edgecolor='white')
axes[2].set_title('Test R² (↑ meilleur)', fontweight='bold')
axes[2].set_ylim(0, 1)
for bar, v in zip(b3, r2s):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('graphiques/comparaison_modeles.png', dpi=150)
plt.show()

In [ ]:
# Prédictions vs réalité pour chaque modèle
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Prédictions vs Réalité', fontsize=13, fontweight='bold')

for ax, nom, pred in zip(axes,
    ['Régression Linéaire', 'Random Forest', 'LightGBM'],
    [pred_lr, pred_rf, pred_lgbm]):
    ax.scatter(y_test, pred, alpha=0.5, color='#5B8DB8', s=20)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--', linewidth=1.5, label='Parfait')
    ax.set_xlabel('Valeur réelle')
    ax.set_ylabel('Valeur prédite')
    ax.set_title(nom)
    ax.legend()

plt.tight_layout()
plt.savefig('graphiques/predictions_vs_realite.png', dpi=150)
plt.show()

In [ ]:
# Importance des features (RandomForest)
importances = pd.Series(
    model_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='#5B8DB8', edgecolor='white')
ax.set_title('Importance des features – Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('graphiques/feature_importance.png', dpi=150)
plt.show()

print('\nTop 3 features les plus importantes :')
print(importances.sort_values(ascending=False).head(3).to_string())

## 7. Interprétation et Conclusion

### Résultats

| Modèle | RMSE | MAE | R² | Interprétation |
|---|---|---|---|---|
| Régression Linéaire | - | - | - | Baseline simple, relations linéaires uniquement |
| Random Forest | - | - | - | Capture les non-linéarités, robuste aux outliers |
| LightGBM | - | - | - | Gradient boosting, généralement meilleure performance |

### Points clés

- **`lstat`** (% population pauvre) et **`rm`** (nb pièces) sont les features les plus prédictives
- La régression linéaire est utile comme baseline mais limitée par les relations non-linéaires
- LightGBM et Random Forest sont plus performants grâce à la capture de relations complexes

### Limites éthiques

- **Dataset obsolète** (années 1970) : les patterns socio-économiques ont changé
- **Variable `b` supprimée** mais d'autres variables comme `lstat` ou `crim` peuvent être des proxies de discrimination
- **Valeurs censurées** à 50k$ : sous-estimation des logements les plus chers
- Ce modèle **ne doit pas être utilisé** pour des décisions réelles sans audit éthique approfondi